In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
import json
import random
import numpy as np
import torch
from datetime import datetime

In [3]:
# Canonical paths
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
preprocessed_dir = f"{base_dir}/AntM2C_preprocessed"
checkpoints_dir = f"{base_dir}/checkpoints"
splits_dir = f"{preprocessed_dir}/splits"
split_features_dir = f"{preprocessed_dir}/split_features"

results_dir = f"{base_dir}/eval_results"
explain_dir = os.path.join(results_dir, "explainability")
os.makedirs(explain_dir, exist_ok=True)

In [5]:
print("--- Paths resolved: ---")
print(" base_dir:", base_dir)
print(" preprocessed_dir:", preprocessed_dir)
print(" checkpoints_dir:", checkpoints_dir)
print(" results_dir:", results_dir)
print(" explain_dir:", explain_dir)

--- Paths resolved: ---
 base_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M
 preprocessed_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed
 checkpoints_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints
 results_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/eval_results
 explain_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/eval_results/explainability


In [6]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Determinism flags (safe defaults)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [8]:
# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


In [9]:
# Load configs + checkpoint + predictions
fusion_config_path = os.path.join(checkpoints_dir, "fusion_config.json")
meta_config_path = os.path.join(checkpoints_dir, "meta_config.json")
model_ckpt_path = os.path.join(checkpoints_dir, "coldfusionx_model_epoch3.pt")

pred_path = os.path.join(results_dir, "test_predictions.npz")

In [10]:
# Load JSON configs
with open(fusion_config_path, "r") as f:
    fusion_config = json.load(f)

with open(meta_config_path, "r") as f:
    meta_config = json.load(f)

# Load predictions (Notebook 09 output)
pred_data = np.load(pred_path)
pred_keys = list(pred_data.keys())

In [11]:
print("--- Loaded Artifacts ---")
print("fusion_config_path:", fusion_config_path)
print("meta_config_path:", meta_config_path)
print("model_ckpt_path:", model_ckpt_path)
print("pred_path:", pred_path)
print("pred_keys:", pred_keys)

# Print key values (audit-friendly)
print("\n--- Key Config Values ---")
print("d_model:", fusion_config.get("d_model"))
print("hidden_dim:", fusion_config.get("hidden_dim"))
print("num_experts:", fusion_config.get("num_experts"))
print("num_scenes:", fusion_config.get("num_scenes"))
print("FEWSHOT_N:", meta_config.get("FEWSHOT_N"))

--- Loaded Artifacts ---
fusion_config_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints/fusion_config.json
meta_config_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints/meta_config.json
model_ckpt_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints/coldfusionx_model_epoch3.pt
pred_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/eval_results/test_predictions.npz
pred_keys: ['y_true', 'y_logit', 'y_prob', 'cold_mask', 'scene_idx', 'alpha', 'gates']

--- Key Config Values ---
d_model: 256
hidden_dim: 256
num_experts: 4
num_scenes: 5
FEWSHOT_N: 10


In [13]:
# Quick sanity on predictions shapes
print("--- Prediction Array Shapes ---")
for k in ["y_true", "y_logit", "y_prob", "cold_mask", "scene_idx", "alpha"]:
    if k in pred_data:
        arr = pred_data[k]
        print(f"{k:10s}: {arr.shape} | dtype={arr.dtype}")

--- Prediction Array Shapes ---
y_true    : (373033,) | dtype=float32
y_logit   : (373033,) | dtype=float32
y_prob    : (373033,) | dtype=float32
cold_mask : (373033,) | dtype=bool
scene_idx : (373033,) | dtype=int64
alpha     : (373033, 1) | dtype=float32


In [ ]:
!git status